<a href="https://colab.research.google.com/github/Shruthi0719/Smart-Traffic-management-System/blob/main/signal_calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics

In [ ]:

import os
import math
import time
import cv2
import torch
import re
import zipfile
import shutil
from google.colab import files

# Load YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')

# Define folders
base_path = os.getcwd()
inputPath = os.path.join(base_path, "test_images")
outputPath = os.path.join(base_path, "output_images")

# Create directories if not exist
os.makedirs(inputPath, exist_ok=True)
os.makedirs(outputPath, exist_ok=True)


print(" Please upload a ZIP file containing your test images:")
uploaded = files.upload()


for filename in uploaded.keys():
    if filename.endswith(".zip"):
        zip_path = os.path.join(base_path, filename)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(inputPath)
        print(f"✅ Extracted files to: {inputPath}")
        os.remove(zip_path)
    else:
        print("⚠️ Please upload a .zip file containing images!")


defaultGreen = 20
defaultYellow = 5
defaultMinimum = 10
defaultMaximum = 60


carTime = 2
busTime = 2.5
truckTime = 2.5
motorcycleTime = 1.5
bicycleTime = 1
noOfLanes = 2


allowed_classes = {"car", "bus", "motorcycle", "truck", "bicycle"}

def get_all_images(path):
    """Recursively find all image files inside a folder."""
    image_list = []
    for root, _, files in os.walk(path):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                image_list.append(os.path.join(root, f))
    return sorted(image_list)

def detectVehicles(filepath):
    """Detect vehicles and save annotated image."""
    filename = os.path.basename(filepath)
    vehicle_counts = {v: 0 for v in allowed_classes}
    img = cv2.imread(filepath, cv2.IMREAD_COLOR)
    results = model(filepath)

    result_labels = results.pandas().xyxy[0]
    for _, vehicle in result_labels.iterrows():
        label = vehicle['name']
        if label in allowed_classes:
            vehicle_counts[label] += 1
            x_min, y_min, x_max, y_max = int(vehicle['xmin']), int(vehicle['ymin']), int(vehicle['xmax']), int(vehicle['ymax'])
            cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
            cv2.putText(img, label, (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    output_file = os.path.join(outputPath, filename)
    cv2.imwrite(output_file, img)

    return vehicle_counts, filename

def calculateGreenTime(vehicle_counts):
    """Compute green signal time from vehicle density."""
    noOfCars = vehicle_counts.get("car", 0)
    noOfMotorcycle = vehicle_counts.get("motorcycle", 0)
    noOfBicycle = vehicle_counts.get("bicycle", 0)
    noOfBuses = vehicle_counts.get("bus", 0)
    noOfTrucks = vehicle_counts.get("truck", 0)

    greenTime = math.ceil(((noOfCars * carTime) + (noOfBicycle * bicycleTime) +
                           (noOfBuses * busTime) + (noOfTrucks * truckTime) +
                           (noOfMotorcycle * motorcycleTime)) / (noOfLanes + 1))
    return max(min(greenTime, defaultMaximum), defaultMinimum)

def calculateRedTime(current_lane_green, all_green_times):
    """Compute red time based on total green cycles."""
    return sum(all_green_times) - current_lane_green + (defaultYellow * (len(all_green_times) - 1))

def processLanes():
    """Process all lanes, handle emergency if present."""
    image_files = get_all_images(inputPath)

    if not image_files:
        print("❌ No images found in extracted folder!")
        return


    emergency_files = [f for f in image_files if re.search(r"amb", os.path.basename(f), re.IGNORECASE)]

    if emergency_files:
        print("🚨 Emergency vehicle detected in images:")
        for f in emergency_files:
            print("   -", os.path.basename(f))

        image_files = emergency_files + [f for f in image_files if f not in emergency_files]
    else:
        print("✅ No emergency vehicle detected. Proceeding with normal flow.")

    signal_times = []
    all_green_times = []

    for i, filepath in enumerate(image_files):
        print(f"\n🔹 Processing Lane {i+1}: {os.path.basename(filepath)}")
        vehicle_counts, filename = detectVehicles(filepath)
        print(f"Detected vehicles: {vehicle_counts}")


        if re.search(r"amb", filename, re.IGNORECASE):
            green_time = defaultMaximum
            print("🚨 Emergency detected — assigning MAX green time!")
        else:
            green_time = calculateGreenTime(vehicle_counts)

        all_green_times.append(green_time)
        red_time = calculateRedTime(green_time, all_green_times)

        print(f"🔸 Signal Timings for Lane {i+1}:")
        print(f"Red: {red_time}s | Green: {green_time}s | Yellow: {defaultYellow}s")

        signal_times.append((red_time, green_time, defaultYellow))
        time.sleep(1)

    print("\n✅ Final Signal Timings Summary:")
    for idx, times in enumerate(signal_times, 1):
        print(f"Lane {idx}: Red = {times[0]}s, Green = {times[1]}s, Yellow = {times[2]}s")


    with open("signal_times.txt", "w") as f:
        for times in signal_times:
            f.write(f"{times[1]}\n")

    print("\n📄 Signal timings saved to 'signal_times.txt'")
    print("🖼️ Annotated images saved in:", outputPath)

if __name__ == "__main__":
    processLanes()


requirements: Ultralytics requirement ['urllib3>=2.6.0 ; python_version > "3.8"'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master


WARNING ⚠️ Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ requirements: ❌ Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
error: Failed to parse: `urllib3>=2.6.0 ; python_version > 3.8`
  Caused by: Expected a quoted string or a valid marker name, found `3.8`
urllib3>=2.6.0 ; python_version > 3.8
                                  ^^^



YOLOv5 🚀 2026-2-4 Python-3.12.12 torch-2.9.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


 Please upload a ZIP file containing your test images:


...

Detected vehicles: {'bus': 5, 'truck': 12, 'bicycle': 1, 'car': 14, 'motorcycle': 1}
🔸 Signal Timings for Lane 6:
Red: 121s | Green: 25s | Yellow: 5s

🔹 Processing Lane 7: 7.jpg


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected vehicles: {'bus': 6, 'truck': 1, 'bicycle': 1, 'car': 10, 'motorcycle': 5}
🔸 Signal Timings for Lane 7:
Red: 151s | Green: 16s | Yellow: 5s

🔹 Processing Lane 8: 8.jpg


/root/.cache/torch/hub/ultralytics_yolov5_master/models/common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


Detected vehicles: {'bus': 3, 'truck': 7, 'bicycle': 0, 'car': 21, 'motorcycle': 1}
🔸 Signal Timings for Lane 8:
Red: 172s | Green: 23s | Yellow: 5s

✅ Final Signal Timings Summary:
Lane 1: Red = 0s, Green = 22s, Yellow = 5s
Lane 2: Red = 27s, Green = 10s, Yellow = 5s
Lane 3: Red = 42s, Green = 24s, Yellow = 5s
Lane 4: Red = 71s, Green = 23s, Yellow = 5s
Lane 5: Red = 99s, Green = 17s, Yellow = 5s
Lane 6: Red = 121s, Green = 25s, Yellow = 5s
Lane 7: Red = 151s, Green = 16s, Yellow = 5s
Lane 8: Red = 172s, Green = 23s, Yellow = 5s

📄 Signal timings saved to 'signal_times.txt'
🖼️ Annotated images saved in: /content/output_images
